# SVD Baseline

## Load Data

In [ ]:
import gdown, zipfile, os

if not os.path.exists('dataset/processed'):
    FILE_ID = '1a7kCwqP2Vhlv43eep0ODdzGVR6v2ouP6'
    gdown.download(id=FILE_ID, output='processed.zip', quiet=False)
    with zipfile.ZipFile('processed.zip', 'r') as z:
        z.extractall('dataset/')
    os.remove('processed.zip')
    print('Downloaded.')
else:
    print('Already exists, skipping download.')

Already exists, skipping download.


In [ ]:
import pandas as pd
import numpy as np
import pickle
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

train_df = pd.read_csv('dataset/processed/train.csv')
val_df   = pd.read_csv('dataset/processed/val.csv')
test_df  = pd.read_csv('dataset/processed/test.csv')
games_df = pd.read_csv('dataset/processed/games_cleaned.csv')

with open('dataset/processed/id_maps.pkl', 'rb') as f:
    maps = pickle.load(f)

n_users = len(maps['user2idx'])
n_items = len(maps['item2idx'])
print(f'n_users: {n_users:,} | n_items: {n_items:,}')
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

n_users: 272,184 | n_items: 21,922
Train: 18,151,978 | Val: 272,184 | Test: 272,184


## Build the User-Item Matrix

In [ ]:
# compute mean rating for each user
user_means = train_df.groupby('user_idx')['Rating'].mean().values
centered_ratings = train_df['Rating'].values - user_means[train_df['user_idx'].values] # mean-center
rows = train_df['user_idx'].values
cols = train_df['item_idx'].values
# mean-centered matrix
R_centered = csr_matrix((centered_ratings.astype(np.float32), (rows, cols)), shape=(n_users, n_items))

In [ ]:
K_FACTORS = 50  # number of latent factors
U, sigma, Vt = svds(R_centered, k=K_FACTORS)

# sort by descending singular value (svds returns in ascending order)
order = np.argsort(-sigma)
U = U[:, order] # (n_users, k)
sigma = sigma[order] # (k,)
Vt = Vt[order, :] # (k, n_items)

# merge sigma and U into 1
U_svd = U * sigma[np.newaxis, :] # (n_users, k)
V_svd = Vt.T # (n_items, k)

print(f'U shape: {U_svd.shape} | V shape: {V_svd.shape}')

U shape: (272184, 50) | V shape: (21922, 50)


In [ ]:
def get_svd_scores(user_idx):
    """
    Predicted score for a user across all items (user x item + user mean rating)
    """
    scores = U_svd[user_idx] @ V_svd.T # (n_items,)
    scores += user_means[user_idx] # add back user bias
    return pd.Series(scores, index=np.arange(n_items))

## Evaluation

Same protocol as all other baselines: 1 positive + 99 random negatives per user, HR@10 / NDCG@10 / MRR@10

In [ ]:
def evaluate(score_fn, train_df, test_df, n_negatives=99, K=10, seed=42, sample_users=1000):
    rng = np.random.default_rng(seed)
    all_items = np.arange(n_items)
    seen = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()
    sampled = test_df.sample(min(sample_users, len(test_df)), random_state=seed)
    hits, ndcgs, mrrs = [], [], []

    for i, (_, row) in enumerate(sampled.iterrows()):
        uid      = int(row['user_idx'])
        pos_item = int(row['item_idx'])
        score_series = score_fn(uid)
        user_seen    = seen.get(uid, set()) | {pos_item}
        candidates   = np.setdiff1d(all_items, list(user_seen))
        neg_items    = rng.choice(candidates, size=n_negatives, replace=False)
        items_to_rank = np.append(neg_items, pos_item)
        scores        = score_series.reindex(items_to_rank).fillna(0).values
        ranked        = items_to_rank[np.argsort(-scores)]
        pos_rank = np.where(ranked == pos_item)[0][0] + 1
        hits.append(1 if pos_rank <= K else 0)
        ndcgs.append(np.log(2) / np.log(pos_rank + 1) if pos_rank <= K else 0)
        mrrs.append(1 / pos_rank if pos_rank <= K else 0)

    return {
        f'HR@{K}':   round(np.mean(hits), 4),
        f'NDCG@{K}': round(np.mean(ndcgs), 4),
        f'MRR@{K}':  round(np.mean(mrrs), 4),
    }

## Results

In [ ]:
svd_results = evaluate(get_svd_scores, train_df, test_df)
all_results = {'SVD': svd_results,}
results_df = pd.DataFrame(all_results).T
print(results_df.to_string())

     HR@10  NDCG@10  MRR@10
SVD  0.192   0.1034  0.0766


## Sample Recommendations

In [ ]:
idx_to_bggid = maps['idx2item']
games_lookup = games_df.set_index('BGGId')[['Name', 'AvgRating']]
seen = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()

def get_recommendations(user_idx, score_fn, top_k=10, model_name='MF'):
    score_series = score_fn(user_idx)
    user_seen = seen.get(user_idx, set())
    score_series[list(user_seen)] = -np.inf
    top_items = score_series.nlargest(top_k).index.tolist()
    bgg_ids = [idx_to_bggid[i] for i in top_items]
    recs = games_lookup.loc[bgg_ids].reset_index()
    recs['SVD'] = score_series[top_items].values
    return recs

# select random user
sample_user = test_df['user_idx'].sample(1).iloc[0]
username = maps['idx2user'][sample_user]
print(f'Top 10 recommendations for user {username}')
recs = get_recommendations(sample_user, get_svd_scores)
print(recs.to_string(index=False))

Top 10 recommendations for user nicoduxx
 BGGId                                          Name  AvgRating      SVD
182028 Through the Ages: A New Story of Civilization    8.38861 8.187272
 31260                                      Agricola    7.92801 8.142200
224517                             Brass: Birmingham    8.66562 8.115371
 84876                       The Castles of Burgundy    8.12711 8.102332
124361                                     Concordia    8.12090 8.064366
171623                     The Voyages of Marco Polo    7.84242 7.908129
164928                                       Orléans    8.07585 7.907660
220308                                  Gaia Project    8.47909 7.857727
182874                           Grand Austria Hotel    7.87875 7.848614
148949                                      Istanbul    7.57049 7.825929
